Latency check test - 01

Latency Benchmark

S3 upload: 7.086 seconds

OCR: 2.851 seconds

Face matching: 0.940 seconds

Liveness: 38.410 seconds

RAG retrieval: 3.843 seconds

LLM assessment: 10.342 seconds

Total KYC time: 63.709 seconds

Liveness is the bottleneck by far.

It takes 38.4 seconds, which is about 60% of our entire 63.7-second pipeline.

TEST 02

Latency Benchmark

S3 upload: 5.807 seconds

OCR: 2.568 seconds

Face matching: 0.692 seconds

Liveness: 13.378 seconds

RAG retrieval: 0.704 seconds

LLM assessment: 9.766 seconds

Total KYC time: 32.994 seconds

We will check same video again for repeatability to check if our measurements are consitent. 

| Stage        |       Test 1 |                Test 2 |      Test 3 |                Test 4 |
| ------------ | -----------: | --------------------: | ----------: | --------------------: |
| Video        | Laptop 24 MB | Phone, multiple faces |  Same phone | **Same laptop 24 MB** |
| S3           |       7.086s |                5.807s |      6.864s |            **8.538s** |
| OCR          |       2.851s |                2.568s |      2.595s |            **2.583s** |
| Face         |       0.940s |                0.692s |      0.694s |            **0.906s** |
| **Liveness** |  **38.410s** |           **13.378s** | **13.416s** |           **38.741s** |
| RAG          |       3.843s |                0.704s |      1.259s |            **1.152s** |
| LLM          |      10.342s |                9.766s |      9.936s |           **10.251s** |
| **Total**    |  **63.709s** |           **32.994s** | **34.842s** |           **62.313s** |


Therefore:

The video itself is causing the difference.

## Latency Benchmark — Baseline

| Test | Video type | S3 Upload | OCR | Face Matching | Liveness | RAG | LLM | Total |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | Laptop, ~24 MB, ~22s | 7.086s | 2.851s | 0.940s | 38.410s | 3.843s | 10.342s | 63.709s |
| 2 | Phone, multiple faces | 5.807s | 2.568s | 0.692s | 13.378s | 0.704s | 9.766s | 32.994s |
| 3 | Same phone video, multiple faces | 6.864s | 2.595s | 0.694s | 13.416s | 1.259s | 9.936s | 34.842s |
| 4 | Same laptop, ~24 MB, ~22s | 8.538s | 2.583s | 0.906s | 38.741s | 1.152s | 10.251s | 62.313s |
| 5 | Phone video | 5.282s | 2.706s | 0.702s | 15.753s | 1.458s | 11.203s | 37.188s |

### Key Observations

- Laptop videos: approximately 62–64 seconds total.
- Phone videos: approximately 33–37 seconds total.
- Liveness is the primary latency bottleneck.
- LLM assessment is the second-largest recurring latency component.
- Face matching and OCR are comparatively fast and stable.
- RAG retrieval is relatively fast after initial model/policy loading.



Imagine **100 users** run your KYC system, and we record how long each request takes.

### Average

**Average = total time of all requests ÷ number of requests.**

For example, if 5 users take:

`30, 32, 34, 36, 38 seconds`

Average = **34 seconds**.

So when we estimated your average at **~35 seconds**, we're saying:

> On average, one phone-video KYC request takes around **35 seconds**.

---

### P50 — Median

**P50 means 50% of requests are faster than this, and 50% are slower.**

If your P50 is **~34–35 seconds**:

> Half of the KYC requests finish in **34–35 seconds or less**, and half take longer.

Think of P50 as your **typical user experience**.

---

### P90

**P90 means 90% of requests finish within this time.**

If P90 is **~38–42 seconds**:

> 90 out of 100 users finish within roughly **38–42 seconds**.

Only the slowest **10 users** take longer.

---

### P95

**P95 means 95% finish within this time.**

If P95 is **~40–45 seconds**:

> 95 out of 100 users finish within roughly **40–45 seconds**.

Only the slowest **5%** take longer.

---

### P99

**P99 means 99% finish within this time.**

If P99 is **~45–55 seconds**:

> 99 out of 100 users finish within roughly **45–55 seconds**.

Only the slowest **1%** take longer.

---

### Easy way to remember 🧠

Imagine **100 KYC customers standing in a line**, ordered from fastest to slowest:

```text
Fastest                                      Slowest
  |                                             |
  ↓                                             ↓
  1 ... 50 ... 90 ... 95 ... 99 ... 100
          ↑      ↑     ↑      ↑
         P50    P90   P95    P99
```

| Metric      | Simple meaning              |
| ----------- | --------------------------- |
| **Average** | Overall typical time        |
| **P50**     | 50% finish within this time |
| **P90**     | 90% finish within this time |
| **P95**     | 95% finish within this time |
| **P99**     | 99% finish within this time |

### Why do engineers care about P95/P99?

Because **average can hide slow users**.

Suppose:

`99 users → 10 seconds`

`1 user → 200 seconds`

Average = about **12 seconds**.

You might say:

> "Our system takes 12 seconds!"

But that one user had a **terrible 200-second experience**.

P99 exposes that tail.

That's why production systems often track **P50 + P95 + P99**, not just average.

For your KYC system, our rough expectation is currently:

**Typical → ~35s**
**90% → ~40s**
**95% → ~40–45s**
**99% → potentially ~45–55s**

Again, those last 5 are **our estimates**, not measured percentiles. we cannot measure with 5-6 test cases we need large representation


| Test  | Video                 |        S3 |       OCR |      Face |   Liveness |       RAG |        LLM |  **Total** |
| ----- | --------------------- | --------: | --------: | --------: | ---------: | --------: | ---------: | ---------: |
| 1     | Laptop ~24 MB         |     7.086 |     2.851 |     0.940 | **38.410** |     3.843 |     10.342 | **63.709** |
| 2     | Phone, multiple faces |     5.807 |     2.568 |     0.692 | **13.378** |     0.704 |      9.766 | **32.994** |
| 3     | Same phone video      |     6.864 |     2.595 |     0.694 | **13.416** |     1.259 |      9.936 | **34.842** |
| 4     | Same laptop ~24 MB    |     8.538 |     2.583 |     0.906 | **38.741** |     1.152 |     10.251 | **62.313** |
| 5     | Phone video           |     5.282 |     2.706 |     0.702 | **15.753** |     1.458 |     11.203 | **37.188** |
| **6** | **Phone video**       | **3.785** | **0.502** | **0.404** | **11.754** | **1.370** | **10.122** | **27.972** |
| 7    | Phone video | 5.735s | 2.564s | 0.706s |  13.009s | 0.699s | 11.311s | **34.104s** |


🐢 Liveness: biggest recurring bottleneck (~12–16s) for phone and for laptop video than around 37-38 seconds

🤖 LLM: second biggest (~10–11s)

☁️ S3: variable (~4–7s)

📄 OCR: generally ~0.5–2.7s

👤 Face matching: <1s

🔎 RAG: ~0.7–1.5s